In [6]:
import warnings
import logging

# 1. Silenciar avisos de versiones de scikit-learn
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

# 2. Silenciar avisos generales de UserWarning (como el de XGBoost)
warnings.filterwarnings("ignore", category=UserWarning)

# 3. Opcional: Silenciar logs internos si se ponen muy ruidosos
logging.getLogger('xgboost').setLevel(logging.ERROR)

In [7]:
import pandas as pd
import joblib
import os
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import warnings
from sklearn.exceptions import InconsistentVersionWarning

# 1. Carga de datos procesados
DATA_PATH = "../data/processed/X_train_balanced_final.csv"
df = pd.read_csv(DATA_PATH)
X_train = df.drop('Revenue', axis=1, errors='ignore')
y_train = df['Revenue']

# 2. Carga de pipeline de preprocesamiento y alineación de variables
PREPROC_PATH = "../models/preprocessing_pipeline.pkl"
preproc_pipeline = joblib.load(PREPROC_PATH)

# Renombrado de columnas para coincidir con lo que espera el pipeline cargado
rename_dict = {
    'Total_PageTypes': 'total_paginas',
    'Total_Duration': 'duracion_total'
}
X_train = X_train.rename(columns=rename_dict)

# Alineación de columnas basada en el entrenamiento original del pipeline
if hasattr(preproc_pipeline, 'feature_names_in_'):
    expected_cols = preproc_pipeline.feature_names_in_
    
    # Verificación de seguridad para asegurar que no falte ninguna columna tras el renombrado
    missing = [col for col in expected_cols if col not in X_train.columns]
    if missing:
        print(f"Advertencia: Aún faltan estas columnas en el DataFrame: {missing}")
    
    X_train = X_train[list(expected_cols)]
    print("Columnas alineadas correctamente con el pipeline.")

# 3. Definición de modelos para establecer el Baseline (parámetros por defecto)
# De acuerdo al Sprint 3, no se realiza optimización de hiperparámetros en esta etapa.
models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000), 
    "Decision_Tree": DecisionTreeClassifier(random_state=42),
    "Random_Forest": RandomForestClassifier(random_state=42),
   "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier()
}

trained_pipelines = {}

# 4. Entrenamiento de modelos integrados en el Pipeline
for name, model in models.items():
    # Integración de las etapas de preprocesamiento con el clasificador actual
    steps = list(preproc_pipeline.steps)
    steps.append(('classifier', model))
    
    final_pipeline = Pipeline(steps)
    final_pipeline.fit(X_train, y_train)
    
    trained_pipelines[name] = final_pipeline
    print(f"Modelo entrenado exitosamente: {name}")

# 5. Persistencia de modelos entrenados
MODELS_DIR = "../models/"
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

for name, pipeline in trained_pipelines.items():
    save_path = os.path.join(MODELS_DIR, f"baseline_{name.lower()}.pkl")
    joblib.dump(pipeline, save_path)
    print(f"Archivo exportado: {save_path}")

Columnas alineadas correctamente con el pipeline.
Modelo entrenado exitosamente: Logistic_Regression
Modelo entrenado exitosamente: Decision_Tree
Modelo entrenado exitosamente: Random_Forest
Modelo entrenado exitosamente: XGBoost
Modelo entrenado exitosamente: SVM
Modelo entrenado exitosamente: KNN
Archivo exportado: ../models/baseline_logistic_regression.pkl
Archivo exportado: ../models/baseline_decision_tree.pkl
Archivo exportado: ../models/baseline_random_forest.pkl
Archivo exportado: ../models/baseline_xgboost.pkl
Archivo exportado: ../models/baseline_svm.pkl
Archivo exportado: ../models/baseline_knn.pkl
